<a href="https://colab.research.google.com/github/2403a52038-star/NLP/blob/main/NLP_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf spacy scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 17.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import pandas as pd
import numpy as np
import re
import fitz  # PDF reader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv("UpdatedResumeDataSet.csv")
df.head()

,Category,Resume
0,Data Science,Skills * Programming Languages: Python (pandas...
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...
2,Data Science,"Areas of Interest Deep Learning, Control Syste..."
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...
4,Data Science,"Education Details \r\n MCA YMCAUST, Faridab..."


In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

df['cleaned_resume'] = df['Resume'].apply(clean_text)


vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['cleaned_resume'])

In [5]:
le = LabelEncoder()
y = le.fit_transform(df['Category'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9430051813471503
Precision: 0.9459006400487656
Recall: 0.9430051813471503
F1 Score: 0.9333962803186132

Classification Report:

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       1.00      1.00      1.00         6
           2       1.00      1.00      1.00         5
           3       1.00      1.00      1.00         7
           4       1.00      1.00      1.00         4
           5       1.00      0.89      0.94         9
           6       1.00      1.00      1.00         5
           7       1.00      1.00      1.00         8
           8       1.00      0.93      0.96        14
           9       1.00      0.20      0.33         5
          10       1.00      1.00      1.00         7
          11       1.00      1.00      1.00         6
          12       1.00      0.92      0.96        12
          13       1.00      1.00      1.00         4
          14       1.00      1.00      1.00      

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [7]:
from google.colab import files
uploaded = files.upload()

Saving vinod(Resume).pdf to vinod(Resume).pdf


In [9]:
def extract_text_from_pdf(file_path):
    text = ""
    doc = fitz.open(file_path)
    for page in doc:
        text += page.get_text()
    return text

resume_text = extract_text_from_pdf(list(uploaded.keys())[0])
resume_text = clean_text(resume_text)

In [10]:
resume_vector = vectorizer.transform([resume_text])

predicted_category = model.predict(resume_vector)
predicted_role = le.inverse_transform(predicted_category)[0]

print("Predicted Job Role:", predicted_role)

Predicted Job Role: Data Science


In [11]:
similarity_scores = cosine_similarity(resume_vector, X)

best_index = similarity_scores.argmax()

print("Best Matching Role:", df.iloc[best_index]['Category'])
print("Similarity Score:", similarity_scores[0][best_index] * 100)

Best Matching Role: Data Science
Similarity Score: 29.618547239742774


In [16]:
skills_list = [
    "python", "java", "sql", "machine learning",
    "deep learning", "nlp", "data analysis"
]

def extract_skills(text):
    return [skill for skill in skills_list if skill in text]

resume_skills = extract_skills(resume_text)

job_desc = """
Looking for a Data Scientist with Python, Machine Learning, NLP, SQL
"""

job_desc = clean_text(job_desc)
jd_skills = extract_skills(job_desc)

matched = set(resume_skills) & set(jd_skills)
missing = set(jd_skills) - set(resume_skills)

score = (len(matched) / len(jd_skills)) * 100 if jd_skills else 0

In [15]:
print("\n===== FINAL RESULT =====")

print("Predicted Role:", predicted_role)
print("Resume Skills:", resume_skills)
print("Required Skills:", jd_skills)

print("Matched Skills:", matched)
print("Missing Skills:", missing)

print("Skill Match Score:", score)
print("Dataset Similarity:", similarity_scores[0][best_index] * 100)


===== FINAL RESULT =====
Predicted Role: Data Science
Resume Skills: ['python', 'java', 'sql', 'machine learning', 'nlp']
Required Skills: ['python', 'sql', 'machine learning', 'nlp']
Matched Skills: {'machine learning', 'nlp', 'python', 'sql'}
Missing Skills: set()
Skill Match Score: 100.0
Dataset Similarity: 29.618547239742774
